In [1]:
import numpy as np

l = np.load("logits_t15/logits_scaled.npy", allow_pickle=True)

x = l[0]  # sample
probs = np.exp(x)

print("blank_frac:", probs[:,0].mean())
print("max_prob:", probs.max())
print("avg max_prob:", probs.max(axis=-1).mean())

blank_frac: 0.6904808
max_prob: 0.99890697
avg max_prob: 0.90810025


In [1]:
import numpy as np

l = np.load("logits.npy", allow_pickle=True)
x = l[0]

probs = np.exp(x)

print("blank_frac:", probs[:,0].mean())
print("avg max_prob:", probs.max(axis=-1).mean())
print("mean gap:", (np.sort(probs, axis=-1)[:,-1] - np.sort(probs, axis=-1)[:,-2]).mean())

blank_frac: 0.6913274
avg max_prob: 0.9870168
mean gap: 0.9747351


In [8]:
import numpy as np

logits = np.load("logits.npy", allow_pickle=True)

# pick a sample
x = logits[0]
probs = np.exp(x)

top1 = probs.argmax(axis=-1)
top1_prob = probs.max(axis=-1)

# second best
sorted_probs = np.sort(probs, axis=-1)
top2_prob = sorted_probs[:, -2]

gap = top1_prob - top2_prob

print("Avg top1 prob:", top1_prob.mean())
print("Avg gap:", gap.mean())

# show some frames
for i in range(20):
    print(f"t={i} | top1_prob={top1_prob[i]:.3f} gap={gap[i]:.3f}")

Avg top1 prob: 0.9870168
Avg gap: 0.9747351
t=0 | top1_prob=1.000 gap=1.000
t=1 | top1_prob=1.000 gap=1.000
t=2 | top1_prob=1.000 gap=1.000
t=3 | top1_prob=1.000 gap=1.000
t=4 | top1_prob=1.000 gap=1.000
t=5 | top1_prob=1.000 gap=1.000
t=6 | top1_prob=1.000 gap=1.000
t=7 | top1_prob=1.000 gap=1.000
t=8 | top1_prob=1.000 gap=1.000
t=9 | top1_prob=1.000 gap=1.000
t=10 | top1_prob=1.000 gap=1.000
t=11 | top1_prob=1.000 gap=1.000
t=12 | top1_prob=1.000 gap=1.000
t=13 | top1_prob=1.000 gap=1.000
t=14 | top1_prob=1.000 gap=1.000
t=15 | top1_prob=1.000 gap=1.000
t=16 | top1_prob=1.000 gap=1.000
t=17 | top1_prob=1.000 gap=1.000
t=18 | top1_prob=1.000 gap=1.000
t=19 | top1_prob=0.999 gap=0.997


In [2]:
import torch

ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')

# Check what's stored in the checkpoint
print("Keys:", ckpt.keys())

# Most training scripts store the best val metric
if 'val_per' in ckpt:
    print(f"Best val PER: {ckpt['val_per']:.4f}")
if 'epoch' in ckpt:
    print(f"Saved at epoch: {ckpt['epoch']}")
if 'loss' in ckpt:
    print(f"Val loss: {ckpt['loss']:.4f}")

Keys: dict_keys(['epoch', 'model', 'ema_model', 'val_PER', 'session_PER', 'args'])
Saved at epoch: 80


In [4]:
import torch
import torch.nn.functional as F
import sys
sys.path.append('/root')  # adjust if your model files are elsewhere

# ── 1. Load args from checkpoint ─────────────────────────────────────────────
ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
args = ckpt['args']
print(f"Saved at epoch : {ckpt['epoch']}")
print(f"Best val PER   : {ckpt['val_PER']:.4f}")
print(f"Args type      : {type(args)}")

# ── 2. Init model using your training args ────────────────────────────────────
# Import whichever file has your BCITransformer class
from model_transformer import BCITransformer   # adjust import to match your file

model = BCITransformer(
    neural_dim  = args.model.n_input_features,
    n_units     = args.model.n_units,
    n_days      = len(args.dataset.sessions),
    n_classes   = args.dataset.n_classes,
    n_layers    = args.model.n_layers,
    patch_size  = args.model.patch_size,
    patch_stride= args.model.patch_stride,
    # add any other args your __init__ needs
)

# ── 3. Load weights ───────────────────────────────────────────────────────────
state = ckpt['model']
# strip _orig_mod. prefix if torch.compile was used
state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
model.load_state_dict(state)
model.eval()
print("Model loaded successfully")

# ── 4. Logit distribution check ───────────────────────────────────────────────
# Use a real sample if you have one, random is fine for distribution shape
sample  = torch.randn(1, 200, 512)   # [B, T, 512]
day_idx = torch.tensor([0])

with torch.no_grad():
    logits = model(sample, day_idx)  # [1, T, 41]
    probs  = logits.softmax(-1)

blank_frac = probs[0, :, 0].mean().item()
max_prob   = probs[0].max(-1).values.mean().item()
entropy    = -(probs * (probs + 1e-9).log()).sum(-1).mean().item()

print(f"\nBlank fraction : {blank_frac:.3f}   (GRU ~0.35, Mamba ~0.76, want <0.55)")
print(f"Mean max prob  : {max_prob:.3f}   (GRU ~0.65, Mamba ~0.98, want <0.80)")
print(f"Mean entropy   : {entropy:.3f}   (higher = less overconfident, want >1.5)")
print(f"\nVal PER        : {ckpt['val_PER']:.4f}")

Saved at epoch : 80
Best val PER   : 0.3064
Args type      : <class 'dict'>


AttributeError: 'dict' object has no attribute 'model'

In [6]:
import torch
import torch.nn.functional as F

ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
args = ckpt['args']

# Print the full args dict so we can see exact key names
import json
print(json.dumps(args, indent=2, default=str))

{
  "data_root": "data",
  "ckpt_dir": "checkpoints_transformer",
  "epochs": 80,
  "batch_size": 8,
  "model_dim": 512,
  "n_layers": 4,
  "n_heads": 8,
  "dim_feedforward": 2048,
  "lr": 0.0001,
  "lr_day": 0.0003,
  "lr_min": 1e-06,
  "warmup_frac": 0.02,
  "decay_frac": 0.8,
  "grad_clip": 1.0,
  "lam": 0.01,
  "dropout": 0.35,
  "smooth_eps": 0.1,
  "ema_decay": 0.999,
  "use_amp": true,
  "aug_static_gain": 0.05,
  "aug_white_noise": 0.01,
  "aug_const_offset": 0.01,
  "aug_random_walk": 0.002,
  "aug_time_mask": 0.1,
  "smooth_std": 2.0,
  "smooth_size": 11,
  "patience": 20,
  "num_workers": 0,
  "max_sessions": null,
  "n_sessions": 124
}


In [7]:
# Since args is a dict, use ['key'] not .key
print("epoch     :", ckpt['epoch'])
print("val_PER   :", ckpt['val_PER'])
print("model keys:", list(args['model'].keys()))
print("dataset keys:", list(args['dataset'].keys()))

print("\npatch_size   :", args['model']['patch_size'])
print("patch_stride :", args['model']['patch_stride'])
print("n_units      :", args['model']['n_units'])
print("n_layers     :", args['model']['n_layers'])
print("n_input_feat :", args['model']['n_input_features'])
print("n_classes    :", args['dataset']['n_classes'])
print("n_days       :", len(args['dataset']['sessions']))

epoch     : 80
val_PER   : 0.30635337717766065


KeyError: 'model'

In [8]:
# Load your actual BCITransformer using the flat args
import sys
sys.path.append('/')

from model_transformer import BCITransformer  # your actual training file

ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
args = ckpt['args']

model = BCITransformer(
    n_sessions     = args['n_sessions'],      # 124
    model_dim      = args['model_dim'],        # 512
    n_layers       = args['n_layers'],         # 4
    n_heads        = args['n_heads'],          # 8
    dim_feedforward= args['dim_feedforward'],  # 2048
    dropout        = args['dropout'],          # 0.35
    n_classes      = 41,                       # phoneme vocab
)

state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}
model.load_state_dict(state)
model.eval()

# Logit distribution
import torch.nn.functional as F
sample  = torch.randn(1, 200, 512)
day_idx = torch.tensor([0])

with torch.no_grad():
    logits = model(sample, day_idx)
    probs  = logits.softmax(-1)

print(f"Blank fraction : {probs[0,:,0].mean().item():.3f}")
print(f"Mean max prob  : {probs[0].max(-1).values.mean().item():.3f}")
print(f"Mean entropy   : {-(probs*(probs+1e-9).log()).sum(-1).mean().item():.3f}")
print(f"Output shape   : {logits.shape}")

TypeError: __init__() got an unexpected keyword argument 'model_dim'

In [9]:
import inspect
from model_transformer import BCITransformer
print(inspect.signature(BCITransformer.__init__))

(self, n_sessions, input_dim=512, n_units=512, n_layers=4, n_heads=8, dim_feedforward=2048, patch_size=16, patch_stride=4, dropout=0.4, proj_dim=128)


In [10]:
import torch
import torch.nn.functional as F
from model_transformer import BCITransformer

ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
args = ckpt['args']

model = BCITransformer(
    n_sessions     = args['n_sessions'],       # 124
    input_dim      = 512,                       # fixed
    n_units        = args['model_dim'],         # 512
    n_layers       = args['n_layers'],          # 4
    n_heads        = args['n_heads'],           # 8
    dim_feedforward= args['dim_feedforward'],   # 2048
    patch_size     = 16,                        # default from signature
    patch_stride   = 4,                         # default from signature
    dropout        = args['dropout'],           # 0.35
)

state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}
model.load_state_dict(state)
model.eval()
print("✅ Model loaded — epoch", ckpt['epoch'], "| val PER", ckpt['val_PER'])

# Logit distribution check
sample  = torch.randn(1, 200, 512)
day_idx = torch.tensor([0])

with torch.no_grad():
    logits = model(sample, day_idx)
    probs  = logits.softmax(-1)

print(f"\nOutput shape   : {logits.shape}")
print(f"Blank fraction : {probs[0,:,0].mean().item():.3f}   (want < 0.55)")
print(f"Mean max prob  : {probs[0].max(-1).values.mean().item():.3f}   (want < 0.80)")
print(f"Mean entropy   : {-(probs*(probs+1e-9).log()).sum(-1).mean().item():.3f}   (want > 1.5)")

✅ Model loaded — epoch 80 | val PER 0.30635337717766065


TypeError: forward() missing 1 required positional argument: 'lengths'

In [11]:
# Check all epoch checkpoints to see the PER curve
import os

checkpoints = sorted([f for f in os.listdir('checkpoints_transformer') if f.startswith('epoch_')])
for ckpt_file in checkpoints:
    c = torch.load(f'checkpoints_transformer/{ckpt_file}', map_location='cpu')
    per = c.get('val_PER', c.get('val_per', '?'))
    epoch = c.get('epoch', ckpt_file)
    print(f"Epoch {epoch:>3} | val PER: {per:.4f}" if isinstance(per, float) else f"{ckpt_file}: {per}")

epoch_005.pt: ?
epoch_010.pt: ?
epoch_015.pt: ?
epoch_020.pt: ?
epoch_025.pt: ?
epoch_030.pt: ?
epoch_035.pt: ?
epoch_040.pt: ?
epoch_045.pt: ?
epoch_050.pt: ?
epoch_055.pt: ?
epoch_060.pt: ?
epoch_065.pt: ?
epoch_070.pt: ?
epoch_075.pt: ?
epoch_080.pt: ?


In [12]:
# Check the actual keys in an epoch checkpoint
c = torch.load('checkpoints_transformer/epoch_080.pt', map_location='cpu')
print("Keys:", c.keys())
for k, v in c.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

Keys: dict_keys(['epoch', 'model'])
  epoch: 80


In [13]:
import inspect
from model_transformer import BCITransformer
print(inspect.signature(BCITransformer.forward))

(self, x, session_ids, lengths)


In [14]:
import torch
import torch.nn.functional as F
from model_transformer import BCITransformer

ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
args = ckpt['args']

model = BCITransformer(
    n_sessions     = args['n_sessions'],
    input_dim      = 512,
    n_units        = args['model_dim'],
    n_layers       = args['n_layers'],
    n_heads        = args['n_heads'],
    dim_feedforward= args['dim_feedforward'],
    patch_size     = 16,
    patch_stride   = 4,
    dropout        = args['dropout'],
)

state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model'].items()}
model.load_state_dict(state)
model.eval()
print("✅ Loaded | epoch", ckpt['epoch'], "| val PER", ckpt['val_PER'])

# Correct forward call
T = 200
sample      = torch.randn(1, T, 512)
session_ids = torch.tensor([0])
lengths     = torch.tensor([T])      # actual length of each sample in batch

with torch.no_grad():
    logits = model(sample, session_ids, lengths)
    probs  = logits.softmax(-1)

print(f"\nOutput shape   : {logits.shape}")
print(f"Blank fraction : {probs[0,:,0].mean().item():.3f}   (want < 0.55)")
print(f"Mean max prob  : {probs[0].max(-1).values.mean().item():.3f}   (want < 0.80)")
print(f"Mean entropy   : {-(probs*(probs+1e-9).log()).sum(-1).mean().item():.3f}   (want > 1.5)")

✅ Loaded | epoch 80 | val PER 0.30635337717766065


AttributeError: 'tuple' object has no attribute 'softmax'

In [15]:
# Check if you have a training log file
import os
for f in os.listdir('.'):
    if f.endswith('.log') or f.endswith('.csv') or f.endswith('.txt') or 'log' in f.lower():
        print(f)

# Also check best.pt fully
ckpt = torch.load('checkpoints_transformer/best.pt', map_location='cpu')
print("\nbest.pt keys:", ckpt.keys())
for k, v in ckpt.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

.hushlogin
logits.npy
logits_output

best.pt keys: dict_keys(['epoch', 'model', 'ema_model', 'val_PER', 'session_PER', 'args'])
  epoch: 80
  val_PER: 0.30635337717766065


In [16]:
with torch.no_grad():
    out = model(sample, session_ids, lengths)

print("Output type:", type(out))
print("Tuple length:", len(out))
for i, o in enumerate(out):
    if isinstance(o, torch.Tensor):
        print(f"  out[{i}] shape: {o.shape}")
    else:
        print(f"  out[{i}] type: {type(o)}")

Output type: <class 'tuple'>
Tuple length: 3
  out[0] shape: torch.Size([47, 1, 41])
  out[1] shape: torch.Size([1, 128])
  out[2] shape: torch.Size([1])


In [17]:
with torch.no_grad():
    out = model(sample, session_ids, lengths)

logits_ctc = out[0]           # [T_out, B, 41]  — time first
out_lengths = out[2]          # [B]  — sequence lengths after patching

# Convert to [B, T, 41] for the distribution check
probs = logits_ctc.permute(1, 0, 2).softmax(-1)   # [B, T_out, 41]

print(f"Logit shape (time-first) : {logits_ctc.shape}")
print(f"Output lengths           : {out_lengths}")
print(f"Blank fraction : {probs[0,:,0].mean().item():.3f}   (want < 0.55)")
print(f"Mean max prob  : {probs[0].max(-1).values.mean().item():.3f}   (want < 0.80)")
print(f"Mean entropy   : {-(probs*(probs+1e-9).log()).sum(-1).mean().item():.3f}   (want > 1.5)")

Logit shape (time-first) : torch.Size([47, 1, 41])
Output lengths           : tensor([47])
Blank fraction : 0.681   (want < 0.55)
Mean max prob  : 0.719   (want < 0.80)
Mean entropy   : 0.914   (want > 1.5)


In [18]:
import torch

# ── Mamba best.pt ────────────────────────────────────────────────────────────
ckpt_mamba = torch.load('checkpoints_phase1/best.pt', map_location='cpu')
print("=== MAMBA ===")
print("Keys:", list(ckpt_mamba.keys()))
for k, v in ckpt_mamba.items():
    if not isinstance(v, dict) and not isinstance(v, list):
        print(f"  {k}: {v}")

# Check args
args_m = ckpt_mamba.get('args', {})
if isinstance(args_m, dict):
    print("\nMamba args:")
    import json
    print(json.dumps(args_m, indent=2, default=str))

=== MAMBA ===
Keys: ['epoch', 'model', 'ema_model', 'opt', 'sched', 'val_PER', 'session_PER', 'args']
  epoch: 80
  val_PER: 0.8224608514710127

Mamba args:
{
  "data_root": "data",
  "ckpt_dir": "checkpoints_phase1",
  "epochs": 80,
  "batch_size": 8,
  "model_dim": 512,
  "n_layers": 4,
  "d_state": 64,
  "lr": 0.0001,
  "lr_day": 0.0003,
  "lr_min": 1e-06,
  "warmup_frac": 0.02,
  "decay_frac": 0.8,
  "grad_clip": 1.0,
  "lam": 0.01,
  "dropout": 0.35,
  "smooth_eps": 0.1,
  "ema_decay": 0.999,
  "use_amp": true,
  "use_compile": false,
  "aug_static_gain": 0.05,
  "aug_white_noise": 0.01,
  "aug_const_offset": 0.01,
  "aug_random_walk": 0.002,
  "aug_time_mask": 0.1,
  "blank_penalty_start": 0.0,
  "blank_penalty_end": 0.0,
  "blank_ramp_epochs": 1,
  "smooth_std": 2.0,
  "smooth_size": 11,
  "patience": 20,
  "num_workers": 0,
  "max_sessions": null,
  "n_sessions": 124
}


In [19]:
from model import MambaDecoder   # your actual mamba model file

args_m = ckpt_mamba['args']
print("\npatch_size  :", args_m.get('patch_size', args_m.get('model', {}).get('patch_size')))
print("patch_stride:", args_m.get('patch_stride', args_m.get('model', {}).get('patch_stride')))

# Then we'll init and check distribution
# First just print the args so I can write the exact init

ModuleNotFoundError: No module named 'mamba_ssm'

In [21]:
import h5py, os

data_root = 'data/hdf5_data_final'

t15_trials, t12_trials = 0, 0

for session in sorted(os.listdir(data_root)):
    train_file = os.path.join(data_root, session, 'data_train.hdf5')
    if not os.path.exists(train_file):
        continue
    with h5py.File(train_file, 'r') as f:
        n = len(f.keys())
    if session.startswith('t15'):
        t15_trials += n
    elif session.startswith('t12'):
        t12_trials += n

total = t15_trials + t12_trials
print(f"T15 train trials : {t15_trials}  ({t15_trials/total*100:.1f}%)")
print(f"T12 train trials : {t12_trials}  ({t12_trials/total*100:.1f}%)")
print(f"Total            : {total}")
print(f"T12/T15 ratio    : {t12_trials/t15_trials:.2f}x")

T15 train trials : 8072  (50.5%)
T12 train trials : 7920  (49.5%)
Total            : 15992
T12/T15 ratio    : 0.98x


In [24]:
import numpy as np

# Load your saved Mamba logits
logits = np.load('logits_output/logits_scaled.npy', allow_pickle=True)

trial = logits[0]  # [T, 41]
probs = np.exp(trial)

print(f'Blank fraction : {probs[:, 0].mean():.3f}')
print(f'Mean max prob  : {probs.max(-1).mean():.3f}')
print(f'Mean entropy   : {-(probs * trial).sum(-1).mean():.3f}')
print(f'Shape          : {trial.shape}')
print(f'Min logit      : {trial.min():.3f}')
print(f'Max logit      : {trial.max():.3f}')

Blank fraction : 0.690
Mean max prob  : 0.908
Mean entropy   : 0.479
Shape          : (224, 41)
Min logit      : -17.063
Max logit      : -0.001


In [25]:
import numpy as np

def rearrange_for_nejm(logits):
    """
    Converts [BLANK, AA, AE, ..., ZH, SIL]  (your order, 41 classes)
    to       [BLANK, SIL, AA, AE, ..., ZH]  (NEJM order)
    
    logits: [T, 41] numpy array (log_probs)
    """
    # your order:  0=BLANK, 1=AA, ..., 39=ZH, 40=SIL
    # nejm order:  0=BLANK, 1=SIL, 2=AA, ..., 40=ZH
    return np.concatenate([
        logits[:, 0:1],    # BLANK stays at 0
        logits[:, 40:41],  # SIL moves from 40 → 1
        logits[:, 1:40],   # AA..ZH shift from 1-39 → 2-40
    ], axis=-1)

# Apply to all trials before saving/sending to NEJM
logits_nejm = np.array([rearrange_for_nejm(t) for t in logits], dtype=object)
np.save('logits_for_nejm.npy', logits_nejm)

In [26]:
import numpy as np

logits = np.load('logits_output/logits.npy', allow_pickle=True)

# Check: after rearrangement, index 1 should be SIL (word boundary)
# SIL should have ~6-7% probability (from your earlier analysis)
trial = logits[0]
probs_before = np.exp(trial)
print(f"Before — idx0(BLANK):{probs_before[:,0].mean():.3f}  idx1:{probs_before[:,1].mean():.3f}  idx40:{probs_before[:,40].mean():.3f}")

trial_nejm = rearrange_for_nejm(trial)
probs_after = np.exp(trial_nejm)
print(f"After  — idx0(BLANK):{probs_after[:,0].mean():.3f}  idx1(SIL):{probs_after[:,1].mean():.3f}  idx2(AA):{probs_after[:,2].mean():.3f}")

# idx1 after should be ~0.067 (your SIL probability from earlier)

Before — idx0(BLANK):0.740  idx1:0.000  idx40:0.105
After  — idx0(BLANK):0.740  idx1(SIL):0.105  idx2(AA):0.000


In [27]:
import numpy as np

logits = np.load('logits_output/logits.npy', allow_pickle=True)
trial = logits[0]
probs = np.exp(trial)

print(f"Shape: {trial.shape}")
print(f"Sum check (should be ~1.0): {probs.sum(-1).mean():.4f}")
print(f"Per-class mean probs (before rearrange):")

PHONEMES = ['BLANK','AA','AE','AH','AO','AW','AY','B','CH','D','DH',
            'EH','ER','EY','F','G','HH','IH','IY','JH','K',
            'L','M','N','NG','OW','OY','P','R','S','SH',
            'T','TH','UH','UW','V','W','Y','Z','ZH','SIL']

for i, ph in enumerate(PHONEMES):
    print(f"  {i:2d} {ph:6s}: {probs[:, i].mean():.4f}")

Shape: (224, 41)
Sum check (should be ~1.0): 1.0000
Per-class mean probs (before rearrange):
   0 BLANK : 0.6905
   1 AA    : 0.0012
   2 AE    : 0.0148
   3 AH    : 0.0104
   4 AO    : 0.0015
   5 AW    : 0.0010
   6 AY    : 0.0012
   7 B     : 0.0011
   8 CH    : 0.0010
   9 D     : 0.0083
  10 DH    : 0.0133
  11 EH    : 0.0029
  12 ER    : 0.0015
  13 EY    : 0.0012
  14 F     : 0.0010
  15 G     : 0.0014
  16 HH    : 0.0022
  17 IH    : 0.0094
  18 IY    : 0.0063
  19 JH    : 0.0011
  20 K     : 0.0109
  21 L     : 0.0087
  22 M     : 0.0011
  23 N     : 0.0158
  24 NG    : 0.0010
  25 OW    : 0.0045
  26 OY    : 0.0081
  27 P     : 0.0059
  28 R     : 0.0016
  29 S     : 0.0173
  30 SH    : 0.0015
  31 T     : 0.0196
  32 TH    : 0.0008
  33 UH    : 0.0015
  34 UW    : 0.0071
  35 V     : 0.0013
  36 W     : 0.0071
  37 Y     : 0.0078
  38 Z     : 0.0072
  39 ZH    : 0.0012
  40 SIL   : 0.0985


In [28]:
import json
with open('wer_results.json') as f:
    results = json.load(f)
print(json.dumps(results, indent=2))

{
  "WER": 0.3887771307570143,
  "failed": 0,
  "model_PER": 10.64,
  "hyps": [
    "you can see the code at this point as well",
    "how does it keep the cost down",
    "not too controversial",
    "the jury in a just work together on it",
    "were quite vocal about it",
    "he said the decision to part ways was mutual",
    "in fact this morning when they were talking",
    "this is like a security risk",
    "he's such a high clay content",
    "current working directory",
    "wait a minute we know this thing isn't",
    "up in new england where i'm from",
    "one thing or the other",
    "he's one of the big proponents of that",
    "i have not heard back lately",
    "to me it's a treasure",
    "he is also a member of the way our economy",
    "i guess i like to take care of it",
    "put that back in the system",
    "he does the u",
    "you just really can't tell what's going to happen",
    "and who is in charge of making that decision",
    "not for the job i have now"

In [29]:
import subprocess, json, itertools

acoustic_scales = [0.4, 0.5, 0.6, 0.8]
blank_penalties = [100, 120, 150, 200]
alphas          = [0.45, 0.55, 0.65]

best_wer  = float('inf')
best_params = {}

for ac, bp, al in itertools.product(acoustic_scales, blank_penalties, alphas):
    cmd = [
        'python', 'run_nejm_decoder.py',
        '--logits_path', 'logits_for_nejm.npy',
        '--lm_path', 'language_model/pretrained_language_models/openwebtext_3gram_lm_sil',
        '--nbest', '50',
        '--acoustic_scale', str(ac),
        '--blank_penalty',  str(bp),
        '--alpha',          str(al),
        '--output_json',    f'wer_ac{ac}_bp{bp}_al{al}.json'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    out_file = f'wer_ac{ac}_bp{bp}_al{al}.json'
    try:
        with open(out_file) as f:
            data = json.load(f)
        wer = data['WER']
        print(f"ac={ac}  bp={bp}  al={al}  →  WER={wer:.4f}")
        if wer < best_wer:
            best_wer = wer
            best_params = {'acoustic_scale': ac, 'blank_penalty': bp, 'alpha': al}
    except:
        print(f"ac={ac}  bp={bp}  al={al}  →  FAILED")

print(f"\nBest WER: {best_wer:.4f}")
print(f"Best params: {best_params}")

ac=0.4  bp=100  al=0.45  →  FAILED
ac=0.4  bp=100  al=0.55  →  FAILED
ac=0.4  bp=100  al=0.65  →  FAILED
ac=0.4  bp=120  al=0.45  →  FAILED
ac=0.4  bp=120  al=0.55  →  FAILED
ac=0.4  bp=120  al=0.65  →  FAILED
ac=0.4  bp=150  al=0.45  →  FAILED
ac=0.4  bp=150  al=0.55  →  FAILED
ac=0.4  bp=150  al=0.65  →  FAILED
ac=0.4  bp=200  al=0.45  →  FAILED
ac=0.4  bp=200  al=0.55  →  FAILED
ac=0.4  bp=200  al=0.65  →  FAILED
ac=0.5  bp=100  al=0.45  →  FAILED
ac=0.5  bp=100  al=0.55  →  FAILED
ac=0.5  bp=100  al=0.65  →  FAILED
ac=0.5  bp=120  al=0.45  →  FAILED
ac=0.5  bp=120  al=0.55  →  FAILED
ac=0.5  bp=120  al=0.65  →  FAILED
ac=0.5  bp=150  al=0.45  →  FAILED
ac=0.5  bp=150  al=0.55  →  FAILED
ac=0.5  bp=150  al=0.65  →  FAILED
ac=0.5  bp=200  al=0.45  →  FAILED
ac=0.5  bp=200  al=0.55  →  FAILED
ac=0.5  bp=200  al=0.65  →  FAILED
ac=0.6  bp=100  al=0.45  →  FAILED
ac=0.6  bp=100  al=0.55  →  FAILED
ac=0.6  bp=100  al=0.65  →  FAILED
ac=0.6  bp=120  al=0.45  →  FAILED
ac=0.6  bp=120  al=0

In [30]:
import subprocess, json, itertools, os

# First check what args run_nejm_decoder.py actually accepts
result = subprocess.run(['python', 'run_nejm_decoder.py', '--help'], 
                        capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

usage: run_nejm_decoder.py [-h] [--logits_dir LOGITS_DIR] [--out OUT]

optional arguments:
  -h, --help            show this help message and exit
  --logits_dir LOGITS_DIR
  --out OUT




In [31]:
result = subprocess.run([
    'python', 'run_nejm_decoder.py',
    '--logits_path', 'logits_for_nejm.npy',
    '--lm_path', 'language_model/pretrained_language_models/openwebtext_3gram_lm_sil',
    '--nbest', '50',
    '--acoustic_scale', '0.5',
    '--blank_penalty', '150',
    '--alpha', '0.55',
    '--output_json', 'test_out.json'
], capture_output=True, text=True)

print("STDOUT:", result.stdout[-2000:])
print("STDERR:", result.stderr[-2000:])

STDOUT: 
STDERR: usage: run_nejm_decoder.py [-h] [--logits_dir LOGITS_DIR] [--out OUT]
run_nejm_decoder.py: error: unrecognized arguments: --logits_path logits_for_nejm.npy --lm_path language_model/pretrained_language_models/openwebtext_3gram_lm_sil --nbest 50 --acoustic_scale 0.5 --blank_penalty 150 --alpha 0.55 --output_json test_out.json



In [33]:
import subprocess, json, os, shutil, time

# The LM server must be restarted with different params for each run
# Check how you launched it originally
result = subprocess.run(['cat', 'nejm-brain-to-text/language_model/language-model-standalone.py'], 
                        capture_output=True, text=True)
print(result.stdout[:3000])

import redis
import argparse
import numpy as np
from datetime import datetime
import time
import os
import re
import logging
import torch
import lm_decoder
from functools import lru_cache
from transformers import AutoModelForCausalLM, AutoTokenizer

# set up logging
logging.basicConfig(format='%(asctime)s %(levelname)s: %(message)s',level=logging.INFO)

# function for initializing the ngram decoder
def build_lm_decoder(
        model_path,
        max_active=7000,
        min_active=200,
        beam=17.,
        lattice_beam=8.0,
        acoustic_scale=1.5,
        ctc_blank_skip_threshold=1.0,
        length_penalty=0.0,
        nbest=1,
    ):

    decode_opts = lm_decoder.DecodeOptions(
        max_active,
        min_active,
        beam,
        lattice_beam,
        acoustic_scale,
        ctc_blank_skip_threshold,
        length_penalty,
        nbest
    )

    TLG_path = os.path.join(model_path, 'TLG.fst')
    words_path = os.path.join(model_path, 'words.txt')
    G_path = os

In [34]:
# Check how you originally ran the LM server
cat ~/nejm-brain-to-text/start_lm.sh
# or check your bash history
history | grep "language-model-standalone"

SyntaxError: invalid syntax (2137118274.py, line 2)

In [35]:
import numpy as np

# Check both logits files to understand which one was used
for fname in ['logits_output/logits_scaled.npy', 'logits_output/logits.npy']:
    try:
        arr = np.load(fname, allow_pickle=True)
        t = arr[0]
        probs = np.exp(t)
        print(f"\n{fname}")
        print(f"  shape[0]: {t.shape}, n_trials: {len(arr)}")
        print(f"  blank_frac: {probs[:,0].mean():.3f}")
        print(f"  SIL(idx40): {probs[:,40].mean():.3f}")
        print(f"  min: {t.min():.3f}  max: {t.max():.3f}")
    except Exception as e:
        print(f"{fname}: {e}")


logits_output/logits_scaled.npy
  shape[0]: (224, 41), n_trials: 1426
  blank_frac: 0.690
  SIL(idx40): 0.098
  min: -17.063  max: -0.001

logits_output/logits.npy
  shape[0]: (224, 41), n_trials: 1426
  blank_frac: 0.740
  SIL(idx40): 0.105
  min: -25.592  max: -0.000


In [37]:
import subprocess, json, os, shutil, time

# Ensure correct directory structure
os.makedirs('logits_sweep', exist_ok=True)
shutil.copy('logits_output/logits_scaled.npy', 'logits_sweep/logits_scaled.npy')
shutil.copy('logits_output/ref_texts.txt',     'logits_sweep/ref_texts.txt')

lm_path = 'language_model/pretrained_language_models/openwebtext_3gram_lm_sil'

# Parameters to sweep
# acoustic_scale: how much to trust acoustics vs LM (higher = trust acoustics more)
# blank_penalty:  ctc_blank_skip_threshold (0-1 range makes more sense than 90)
#                 it's the blank prob threshold BELOW which frames are not skipped
# alpha:          LM weight
params = [
    # acoustic_scale, blank_penalty, alpha   ← label
    (0.325, 90,  0.55),   # original — your 38.9% baseline
    (0.5,   90,  0.55),   # more acoustic trust
    (0.8,   90,  0.55),   # even more acoustic
    (1.0,   90,  0.55),   # full acoustic
    (0.325, 50,  0.55),   # lower blank skip threshold
    (0.325, 30,  0.55),   # even lower
    (0.5,   50,  0.55),
    (0.5,   30,  0.55),
    (0.8,   50,  0.55),
    (0.5,   90,  0.45),   # less LM
    (0.5,   90,  0.65),   # more LM
    (0.8,   50,  0.45),
]

results = []

for ac, bp, al in params:
    tag      = f"ac{ac}_bp{bp}_al{al}"
    out_file = f"wer_{tag}.json"

    if os.path.exists(out_file):
        with open(out_file) as f:
            d = json.load(f)
        print(f"[cached] ac={ac} bp={bp} al={al} → WER={d['WER']*100:.2f}%")
        results.append((d['WER'], ac, bp, al))
        continue

    print(f"\n{'='*55}")
    print(f"▶ ac={ac}  bp={bp}  al={al}")

    # Kill old LM server
    subprocess.run(['pkill', '-f', 'language-model-standalone'], capture_output=True)
    time.sleep(3)

    # Start new LM server
    lm_proc = subprocess.Popen([
        'python',
        'language_model/language-model-standalone.py',
        '--lm_path',        lm_path,
        '--nbest',          '50',
        '--acoustic_scale', str(ac),
        '--blank_penalty',  str(bp),
        '--alpha',          str(al),
        '--redis_ip',       'localhost',
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print("  Waiting 20s for LM server...")
    time.sleep(20)

    # Run decoder
    proc = subprocess.run([
        'python', 'run_nejm_decoder.py',
        '--logits_dir', 'logits_sweep',
        '--out',        out_file,
    ], capture_output=True, text=True, timeout=900)

    # Parse WER from stdout even if JSON fails
    wer = None
    for line in proc.stdout.split('\n'):
        if 'FINAL WER' in line or 'WER=' in line:
            print(' ', line.strip())
        if 'WER:' in line:
            try:
                wer = float(line.split(':')[1].strip().replace('%','')) / 100
            except: pass

    try:
        with open(out_file) as f:
            d = json.load(f)
        wer = d['WER']
        print(f"  ✅ WER = {wer*100:.2f}%")
        results.append((wer, ac, bp, al))
    except:
        print(f"  ❌ No output JSON. stdout tail:\n{proc.stdout[-500:]}")

    lm_proc.terminate()
    time.sleep(2)

# Final summary
print("\n" + "="*55)
print("SWEEP RESULTS (sorted by WER):")
for wer, ac, bp, al in sorted(results):
    marker = " ← BEST" if wer == min(r[0] for r in results) else ""
    print(f"  ac={ac:<5} bp={bp:<4} al={al}  →  WER={wer*100:.2f}%{marker}")


▶ ac=0.325  bp=90  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ ac=0.5  bp=90  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ ac=0.8  bp=90  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ ac=1.0  bp=90  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ ac=0.325  bp=50  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ ac=0.325  bp=30  al=0.55
  Waiting 20s for LM server...
  ❌ No output JSON. stdout tail:
Loading logits...
Loaded 1426 samples, shape: (224, 41)
blank_frac: 0.690


▶ a